In [ ]:
import os
import gc
import polars as pl
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# ── Paths ────────────────────────────────────────────────────
INPUT_PATH  = "/kaggle/input/datasets/shinnraa/data-train-amazon/processed/train_text_for_sentiment.parquet"
OUTPUT_PATH = "/kaggle/working/review_sentiment.parquet"
CKPT_PATH   = "/kaggle/working/sentiment_checkpoint.parquet"
CHUNK_DIR   = "/kaggle/working/chunks"
os.makedirs(CHUNK_DIR, exist_ok=True)
# ── Config ───────────────────────────────────────────────────
MODEL_NAME   = "cardiffnlp/twitter-roberta-base-sentiment-latest"
MAX_LEN      = 128     # review thời trang ngắn, 128 đủ dùng
BATCH_SIZE   = 256     # mỗi GPU nhận 128 samples (DataParallel chia đôi)
CHECKPOINT_EVERY = 500_000  # lưu checkpoint mỗi N rows để tránh Kaggle timeout
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
NUM_GPUS     = torch.cuda.device_count()
print(f"GPUs available: {NUM_GPUS} | Device: {DEVICE}")

In [ ]:
# ════════════════════════════════════════════════════════════
# 1. LOAD & CLEAN
# ════════════════════════════════════════════════════════════

df = pl.read_parquet(INPUT_PATH)

print(f"Rows trước khi clean : {len(df):,}")
original_len = len(df) # Thêm biến lưu chiều dài gốc

df = (
    df
    .filter(pl.col("text").is_not_null())
    .with_columns(
        pl.col("text").str.replace_all(r"<[^>]+>", " ")
    )
    .with_columns(
        pl.col("text").str.replace_all(r"[\n\r\t\xa0]+", " ")
        .str.replace_all(r"\s{2,}", " ") 
    )
    .filter(pl.col("text").str.to_lowercase() != "no review")
)
print(f"Rows sau khi clean   : {len(df):,}")
print(f"Rows đã drop         : {original_len - len(df):,}")

# Lấy list text và metadata để inference
texts    = df["text"].to_list()
user_ids = df["mapped_user_id"].to_list()
item_ids = df["mapped_item_id"].to_list()
ratings  = df["rating"].to_list()

del df
gc.collect()

In [ ]:
# ════════════════════════════════════════════════════════════
# 2. DATASET CLASS
# ════════════════════════════════════════════════════════════

class ReviewDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


In [ ]:
# ════════════════════════════════════════════════════════════
# 3. LOAD MODEL + TOKENIZER
# ════════════════════════════════════════════════════════════

print("Loading tokenizer & model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Lấy mapping label tự động từ model config để code "chống đạn"
id2label = model.config.id2label
print(f"Model Label Mapping: {id2label}")

# Tìm vị trí chính xác của từng nhãn thay vì gán cứng
# Giả sử tên nhãn trong model config chứa từ khóa 'neg', 'neu', 'pos'
NEG_IDX = next(k for k, v in id2label.items() if 'neg' in v.lower())
NEU_IDX = next(k for k, v in id2label.items() if 'neu' in v.lower())
POS_IDX = next(k for k, v in id2label.items() if 'pos' in v.lower())
assert all(idx is not None for idx in [NEG_IDX, NEU_IDX, POS_IDX]), \
    f"Không tìm thấy đủ 3 nhãn trong model config: {id2label}"
print(f"Indices -> Negative: {NEG_IDX}, Neutral: {NEU_IDX}, Positive: {POS_IDX}")

# DataParallel tự động chia batch cho 2 GPU
if NUM_GPUS > 1:
    model = nn.DataParallel(model)

model = model.to(DEVICE)
model.eval()
print(f"Model loaded | DataParallel: {NUM_GPUS > 1}")

In [ ]:
import torch.nn.functional as F

# ════════════════════════════════════════════════════════════
# 4. INFERENCE VỚI CHECKPOINT
# ════════════════════════════════════════════════════════════

# Kiểm tra nếu đã có checkpoint (resume khi bị timeout)
start_idx     = 0

# Tìm tất cả các file chunk đã lưu trong CHUNK_DIR
existing_chunks = [f for f in os.listdir(CHUNK_DIR) if f.endswith(".parquet")]

if len(existing_chunks) > 0:
    # Đọc lại toàn bộ các chunk đã lưu để lấy số dòng đã xử lý
    ckpt_df = pl.scan_parquet(f"{CHUNK_DIR}/*.parquet").collect()
    start_idx  = len(ckpt_df)
    print(f"Resume từ {len(existing_chunks)} chunks: {start_idx:,} rows đã xử lý")
    del ckpt_df
    gc.collect()
else:
    print("Bắt đầu inference từ đầu...")

# Chỉ inference phần chưa xử lý
remaining_texts = texts[start_idx:]

dataset = ReviewDataset(remaining_texts, tokenizer, MAX_LEN)
loader  = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# KHỞI TẠO current_idx BẰNG start_idx
current_idx = start_idx 

batch_prob_neg = []
batch_prob_neu = []
batch_prob_pos = []

with torch.no_grad():
    for batch_idx, batch in enumerate(loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            
        # Chuyển logits thành xác suất (Softmax)
        probs = F.softmax(logits, dim=-1).cpu().numpy().astype(np.float16)
        
        # Tách ra 3 mảng xác suất
       # ✅ Dùng index động từ cell 3
        batch_prob_neg.extend(probs[:, NEG_IDX].tolist())
        batch_prob_neu.extend(probs[:, NEU_IDX].tolist())
        batch_prob_pos.extend(probs[:, POS_IDX].tolist())
        
        # Cộng dồn số lượng row đã xử lý
        batch_size_current = len(probs)
        current_idx += batch_size_current 

        # ── Checkpoint mỗi CHECKPOINT_EVERY rows ──
        if len(batch_prob_neg) >= CHECKPOINT_EVERY:
            chunk_id = len([f for f in os.listdir(CHUNK_DIR) if f.endswith(".parquet")])
            chunk_path = f"{CHUNK_DIR}/part_{chunk_id}.parquet"
            
            # Tính toán vị trí cắt mảng tuyệt đối
            start_idx_chunk = current_idx - len(batch_prob_neg)
            
            # Ghi DataFrame với 3 cột xác suất
            ckpt_df = pl.DataFrame({
                "mapped_user_id"  : user_ids[start_idx_chunk : current_idx],
                "mapped_item_id"  : item_ids[start_idx_chunk : current_idx],
                "rating"          : ratings[start_idx_chunk : current_idx],
                "prob_neg"        : pl.Series(batch_prob_neg, dtype=pl.Float32),
                "prob_neu"        : pl.Series(batch_prob_neu, dtype=pl.Float32),
                "prob_pos"        : pl.Series(batch_prob_pos, dtype=pl.Float32),
            })
            ckpt_df.write_parquet(chunk_path)
            print(f" Đã lưu {chunk_path} ({current_idx:,} / {len(texts):,} rows)")
            
            # Reset lại cả 3 list để giải phóng RAM
            batch_prob_neg = []
            batch_prob_neu = []
            batch_prob_pos = []
            gc.collect()

# Xử lý phần chunk cuối cùng (Tail)
if len(batch_prob_neg) > 0:
    # Đã sửa: Cấp ID an toàn dựa trên số file thực tế đang có
    chunk_id = len([f for f in os.listdir(CHUNK_DIR) if f.endswith(".parquet")]) + 1
    chunk_path = f"{CHUNK_DIR}/part_{chunk_id}_tail.parquet"
    
    start_idx_chunk = current_idx - len(batch_prob_neg)
    
    ckpt_df = pl.DataFrame({
        "mapped_user_id"  : user_ids[start_idx_chunk : current_idx],
        "mapped_item_id"  : item_ids[start_idx_chunk : current_idx],
        "rating"          : ratings[start_idx_chunk : current_idx],
        "prob_neg"        : pl.Series(batch_prob_neg, dtype=pl.Float32),
        "prob_neu"        : pl.Series(batch_prob_neu, dtype=pl.Float32),
        "prob_pos"        : pl.Series(batch_prob_pos, dtype=pl.Float32),
    })
    
    ckpt_df.write_parquet(chunk_path)
    print(f" Đã lưu chunk cuối cùng {chunk_path} ({current_idx:,} / {len(texts):,} rows)")
    
    batch_prob_neg = []
    batch_prob_neu = []
    batch_prob_pos = []
    gc.collect()

print(f"\nInference hoàn tất toàn bộ: {current_idx:,} / {len(texts):,} rows")

In [ ]:
# ════════════════════════════════════════════════════════════
# 5. GOM CHUNKS VÀ XUẤT OUTPUT (CONCATENATION)
# ════════════════════════════════════════════════════════════
import re
import shutil
print("Đang gom các chunks...")

# 1. Lấy danh sách file và sắp xếp chuẩn xác theo số thứ tự
chunk_files = [f for f in os.listdir(CHUNK_DIR) if f.endswith(".parquet")]

def get_chunk_id(filename):
    match = re.search(r"part_(\d+)", filename)
    return int(match.group(1)) if match else -1

chunk_files.sort(key=get_chunk_id)

# 2. Ghép nối
chunk_paths = [os.path.join(CHUNK_DIR, f) for f in chunk_files]
lazy_frames = [pl.scan_parquet(p) for p in chunk_paths]

final_df = pl.concat(lazy_frames).collect()

# 3. Lưu file
final_df.write_parquet(OUTPUT_PATH)

print(f"\nOutput cuối cùng đã lưu tại: {OUTPUT_PATH}")
print(f"Shape tổng cộng: {final_df.shape}")
# Bỏ dòng in value_counts() của sentiment_label vì cột này không còn tồn tại

if os.path.exists(CHUNK_DIR):
    shutil.rmtree(CHUNK_DIR)
    print("\nĐã dọn dẹp thư mục chứa các chunks tạm thời.")

In [ ]:
# ════════════════════════════════════════════════════════════
# 6. SANITY CHECK
# ════════════════════════════════════════════════════════════

# Kiểm tra null cho cả 3 cột xác suất
for col in ["prob_neg", "prob_neu", "prob_pos"]:
    assert final_df[col].null_count() == 0, f"FAIL: Cột {col} có giá trị null"

# Thay thế result_df bằng final_df
assert len(final_df) == len(texts), \
    f"FAIL: row count mismatch — expected {len(texts)}, got {len(final_df)}"

print("\nSanity check: PASS")